# AgentCore Gateway — MCP 세션 관리

## 개요

MCP 세션을 사용하면 클라이언트와 AgentCore Gateway 간에 상태를 유지하는 상호 작용이 가능합니다. 세션을 활성화하면 게이트웨이는 초기화 중에 고유한 세션 식별자를 생성하고 여러 요청에 걸쳐 상태를 유지하여 elicitation 및 sampling과 같은 고급 MCP 기능을 사용할 수 있게 합니다.


### 게이트웨이에서 세션 활성화

세션을 활성화하려면 게이트웨이를 생성하거나 업데이트할 때 protocolConfiguration.mcp 필드에 sessionConfiguration을 지정합니다.


```bash
{
  "protocolConfiguration": {
    "mcp": {
      "sessionConfiguration": {
        "sessionTimeoutInSeconds": 3600
      }
    }
  }
}
```

sessionTimeoutInSeconds 파라미터는 선택 사항입니다. 생략하면 기본 제한 시간은 3600초(1시간)입니다. 유효 범위는 900초(15분)부터 28800초(8시간)까지입니다. 제한 시간은 첫 번째 initialize 요청부터 계산되는 절대 시간입니다.

![세션 다이어그램 자리 표시자](./images/session-management.png)

참고: 게이트웨이에서 세션을 활성화한 경우 게이트웨이 대상의 헤더 전파 설정에 있는 metadataConfiguration에 Mcp-Session-Id를 포함할 수 없습니다. 게이트웨이가 세션 ID를 내부적으로 관리합니다. 이를 포함하려고 하면 HTTP 400 Bad Request 오류가 반환됩니다.

### 세션 사용의 이점


1.  상태를 유지하는 MCP 서버 대상 상호 작용: 
    게이트웨이는 MCP 서버 대상의 세션 ID를 저장하고 이후 도구 호출에서 재사용합니다. 따라서 요청마다 다시 초기화하지 않아도 되며 대상이 여러 호출에 걸쳐 컨텍스트를 유지할 수 있습니다.

2.  AgentCore Runtime 대상의 응답 속도 향상: 
    대상의 세션을 재사용하면 AgentCore Runtime이 요청마다 새로운 MCP 서버 연결을 콜드 스타트할 필요가 없어 응답 시간이 단축됩니다.

3.  고급 MCP 기능 활성화: 
    여러 요청에 걸쳐 상태를 추적해야 하는 elicitation 및 sampling 기능을 사용하려면 세션이 필요합니다.

4.  사용자 범위 보안(인증된 게이트웨이): 
    인바운드 인증을 사용하는 게이트웨이에서는 세션이 검증된 사용자 자격 증명에 바인딩되어 세션 하이재킹을 방지합니다.


## 워크숍 로드맵

| 단계 | 수행 내용 |
|---|---|
| **1** | Notebook을 설정합니다(환경 변수, 유틸리티, 로깅). |
| **2** | 게이트웨이를 생성합니다: Cognito 인바운드 인증, IAM 역할, 세션이 활성화된 게이트웨이. |
| **3** | `labsession` FastMCP 서버를 AgentCore Runtime에 배포합니다. |
| **4** | 게이트웨이 대상으로 연결합니다(아웃바운드 OAuth, 대상 생성, 인바운드 토큰). |
| **5** | 세션을 초기화하고 `Mcp-Session-Id` 헤더를 확인합니다. |
| **6** | 세션 연속성을 확인합니다. 한 세션 내에서 호출할 때마다 `session_counter`가 증가합니다. |
| **7** | 세션 격리를 확인합니다. 새 `initialize`는 새로운 ID를 반환하고 새로운 카운터를 시작합니다. |
| **8** | 세션 ID 오류 규약을 확인합니다. `Mcp-Session-Id`가 없거나 유효하지 않은 경우를 살펴봅니다. |
| **9** | 하이재킹 방지 동작을 살펴봅니다. 세션은 인증된 자격 증명 범위로 제한됩니다. |
| **10** | 리소스를 정리합니다. |

## 튜토리얼 세부 정보

| 항목 | 세부 정보 |
|:---|:---|
| 튜토리얼 유형 | 대화형 |
| AgentCore 구성 요소 | AgentCore Gateway, AgentCore Identity, AgentCore Runtime |
| 게이트웨이 대상 유형 | MCP 서버 |
| 게이트웨이 기능 | 세션 활성화, 스트리밍 비활성화, 인터셉터 없음 |
| MCP 전송 방식 | Streamable HTTP, 단일 JSON 응답 |
| 인바운드 인증 | Cognito (M2M) |
| 아웃바운드 인증 | OAuth2 자격 증명 공급자를 통한 Cognito (M2M) |
| 사용 SDK | boto3 + raw httpx |


### 1단계: 설정 및 사전 요구 사항

Jupyter(Python 3.10+ 커널), Node.js + npm(AgentCore CLI용), `us-west-2`로 구성된 AWS 자격 증명과 CloudFormation, Cognito IDP, IAM 및 Bedrock AgentCore(제어 + 런타임)에 대한 IAM 권한이 필요합니다.

In [ ]:
# 현재 디렉터리의 requirements.txt 또는 pyproject.toml에서 설치
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
!npm install -g @aws/agentcore

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# 이 Notebook에서 사용할 import 및 상수
import utils
import logging
import boto3
import json

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

REGION = boto3.Session().region_name
COGNITO_STACK_NAME = "agentcore-gateway-lab"
TEMPLATE_PATH = "cloudformation/cognito-signup-stack.yaml"
MCP_SERVER_NAME = "lab5session"
GATEWAY_NAME = "ac-gateway-sessions-only"

cfn = boto3.client("cloudformation", region_name=REGION)
cognito = boto3.client("cognito-idp", region_name=REGION)
print("REGION:", REGION)

### 2단계: 게이트웨이 생성

### 2.1단계: CloudFormation을 통해 Cognito 배포

In [ ]:
outputs = utils.deploy_cognito_stack(cfn, COGNITO_STACK_NAME, TEMPLATE_PATH)

# 게이트웨이 인바운드
gw_user_pool_id = outputs["UserPoolId"]
gw_client_id = outputs["GatewayClientId"]
gw_cognito_discovery_url = outputs["DiscoveryUrl"]
scopeString = outputs["GatewayScope"]
token_endpoint = outputs["TokenEndpoint"]
gw_client_secret = cognito.describe_user_pool_client(
    UserPoolId=gw_user_pool_id, ClientId=gw_client_id
)["UserPoolClient"]["ClientSecret"]

# MCP 서버로의 아웃바운드(동일한 풀)
runtime_client_id = outputs["MCPClientId"]
runtime_cognito_discovery_url = gw_cognito_discovery_url
runtimeScopeString = outputs["MCPScope"]
runtime_client_secret = cognito.describe_user_pool_client(
    UserPoolId=gw_user_pool_id, ClientId=runtime_client_id
)["UserPoolClient"]["ClientSecret"]

print(f"User Pool ID:       {gw_user_pool_id}")
print(f"Discovery URL:      {gw_cognito_discovery_url}")
print(f"Token endpoint:     {token_endpoint}")
print(f"Gateway client ID:  {gw_client_id}")
print(f"MCP client ID:      {runtime_client_id}")
print(f"Gateway scope:      {scopeString}")
print(f"MCP scope:          {runtimeScopeString}")

### 2.2단계: 게이트웨이 IAM 역할 생성

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role_with_region(
    GATEWAY_NAME, REGION
)
print("AgentCore Gateway role ARN:", agentcore_gateway_iam_role["Role"]["Arn"])

### 2.3단계: 세션 전용 구성으로 게이트웨이 생성

`sessionConfiguration.sessionTimeoutInSeconds: 3600`(1시간, 기본값, 유효 범위 `[900, 28800]`). `streamingConfiguration`은 **없으며** 스트리밍은 비활성화됩니다. `interceptorConfigurations`도 **없습니다**.

In [ ]:
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [gw_client_id],
        "discoveryUrl": gw_cognito_discovery_url,
    }
}

create_response = gateway_client.create_gateway(
    name=GATEWAY_NAME,
    roleArn=agentcore_gateway_iam_role["Role"]["Arn"],
    protocolType="MCP",
    protocolConfiguration={
        "mcp": {
            "supportedVersions": ["2025-11-25"],
            "sessionConfiguration": {"sessionTimeoutInSeconds": 3600},
        }
    },
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="Sessions-only gateway (no streaming, no interceptor)",
)
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(f"Gateway ID:  {gatewayID}")
print(f"Gateway URL: {gatewayURL}")

### 3단계: AgentCore Runtime에 MCP 서버 배포

### 3.1단계: MCP 서버 코드 확인

`labsession`은 의도적으로 최소한으로 구성되어 있습니다. `session_counter`는 `Mcp-Session-Id`를 키로 사용하여 세션별 호출 횟수를 유지하며, 정상 동작 확인용 도구로 `getOrder`와 `updateOrder`도 제공합니다.

In [ ]:
from IPython.display import Code

Code("mcpservers/app/labsession/main.py", language="python")

### 3.2단계: 에이전트 등록

In [ ]:
!cd mcpservers && agentcore add agent \
    --name {MCP_SERVER_NAME} \
    --type byo \
    --language Python \
    --protocol MCP \
    --code-location app/labsession \
    --authorizer-type CUSTOM_JWT \
    --discovery-url {runtime_cognito_discovery_url} \
    --allowed-clients {runtime_client_id} \
    --allowed-scopes {runtimeScopeString}

### 3.3단계: AgentCore CLI를 통해 배포

In [ ]:
!cd mcpservers && agentcore deploy

In [ ]:
agent = utils.get_agent_status(MCP_SERVER_NAME)

mcp_arn = agent["identifier"]
mcp_url = agent["invocationUrl"]
mcp_id = mcp_arn.split("/")[-1]

print(f"mcp_arn: {mcp_arn}")
print(f"mcp_id:  {mcp_id}")
print(f"mcp_url: {mcp_url}")

### 4단계: MCP 서버를 게이트웨이 대상으로 연결

### 4.1단계: 아웃바운드 OAuth2 자격 증명 공급자

In [ ]:
identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

cognito_provider = identity_client.create_oauth2_credential_provider(
    name=f"{GATEWAY_NAME}-identity",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "oauthDiscovery": {"discoveryUrl": runtime_cognito_discovery_url},
            "clientId": runtime_client_id,
            "clientSecret": runtime_client_secret,
        }
    },
)
cognito_provider_arn = cognito_provider["credentialProviderArn"]
print(cognito_provider_arn)

### 4.2단계: 게이트웨이 대상 생성

In [ ]:
create_gateway_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target",
    gatewayIdentifier=gatewayID,
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": mcp_url}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
    # `Mcp-Session-Id`는 허용 목록에 포함할 수 없음
    # metadataConfiguration={
    #     "allowedRequestHeaders": ["Mcp-Session-Id"],
    #     "allowedResponseHeaders": ["Mcp-Session-Id"],
    # },
)
gatewayTargetID = create_gateway_target_response["targetId"]
print(f"Created target: {gatewayTargetID}")

### 4.3단계: 대상이 READY 상태인지 확인

In [ ]:
list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID)
print(json.dumps(list_targets_response, default=str, indent=2))

### 4.4단계: 인바운드 액세스 토큰 가져오기

In [ ]:
token_response = utils.get_token(
    token_endpoint, gw_client_id, gw_client_secret, scopeString
)
token = token_response["access_token"]
print("Token (truncated):", token[:60], "...")

## 5단계: 초기화하고 `Mcp-Session-Id` 캡처

세션이 활성화된 게이트웨이는 `Mcp-Session-Id`를 첫 번째 `initialize`에 대한 응답으로 발급합니다. 

In [ ]:
from gateway_mcp_client import GatewayMCPClient


def _get_inbound_token() -> str:
    return utils.get_token(token_endpoint, gw_client_id, gw_client_secret, scopeString)[
        "access_token"
    ]


# 세션 A: 새로운 GatewayMCPClient. .initialize()를 호출하면 게이트웨이가 응답으로
# 발급한 `Mcp-Session-Id`를 캡처하고 이후 모든 요청에 자동으로 포함함
mcp_a = GatewayMCPClient(gatewayURL, _get_inbound_token)
init_a = mcp_a.initialize(client_info={"name": "session-demo", "version": "0.1"})
print(f"init A HTTP {init_a['http_status']}")
print(f"Session A id: {mcp_a.session_id}")

## 6단계: 세션 연속성 - 카운터 증가

`session_counter`는 `{session_id, count}`를 반환합니다. 여기서 `session_id`는 fastmcp의 프로세스별 *업스트림* 세션 ID이며, `count`는 동일한 세션 내에서 호출할 때마다 증가합니다. 같은 `Mcp-Session-Id`로 반복 호출하면 호출 횟수가 증가하는 것을 확인할 수 있습니다.


In [ ]:
def call_session_counter(client, label):
    msg = client.call_tool("mcp-server-target___session_counter", {})
    print(f"  [{label}] result={msg.get('result', {}).get('structuredContent')}")
    return msg


print("Three calls on session A:")
call_session_counter(mcp_a, "A.1")
call_session_counter(mcp_a, "A.2")
call_session_counter(mcp_a, "A.3")

In [ ]:
call_session_counter(mcp_a, "A.4")

## 7단계: 세션 격리 - 새 `initialize`로 카운터 재설정

동일한 클라이언트에서 두 번째로 `initialize`를 호출하면 다른 `Mcp-Session-Id`가 반환됩니다. 이 새로운 ID로 호출하면 `count=1`부터 시작하며, 상태와 업스트림 세션이 각각 분리됩니다.


In [ ]:
# 세션 B: 두 번째 GatewayMCPClient. initialize()는 다른 Mcp-Session-Id를 반환하며
# 업스트림 서버의 세션별 카운터가 처음부터 시작됨
mcp_b = GatewayMCPClient(gatewayURL, _get_inbound_token)
mcp_b.initialize(client_info={"name": "session-demo", "version": "0.1"})
print(f"Session B id: {mcp_b.session_id}  (distinct from A: {mcp_a.session_id})")
print()

print("Two calls on session B:")
call_session_counter(mcp_b, "B.1")
call_session_counter(mcp_b, "B.2")
print()
print("Then back to session A — count continues from where it left off:")
call_session_counter(mcp_a, "A.5")

## 8단계: 세션 ID 오류 규약

게이트웨이의 세션 검증기는 세션 조회가 실패하는 다음 세 가지 경우를 처리합니다.

| 테스트 | 예상 결과 | 본문 |
|---|---|---|
| `Mcp-Session-Id` 누락 | HTTP 400 | `Missing required Mcp-Session-Id header` |
| 임의로 생성되었거나 발급된 적 없는 `Mcp-Session-Id` | HTTP 404 | `Session not found or expired` |
| 인증된 다른 자격 증명에서 발급된 실제 `Mcp-Session-Id` | HTTP 404 | `Session not found or expired` |


In [ ]:
import uuid

# 테스트 1: 한 번도 초기화하지 않은 클라이언트 - Mcp-Session-Id 헤더 없음
mcp_no_sid = GatewayMCPClient(gatewayURL, _get_inbound_token)
r = mcp_no_sid.rpc_raw("tools/list")
print(f"NO Mcp-Session-Id           -> HTTP {r.status_code}  body={r.text[:120]!r}")

# 테스트 2: 게이트웨이가 발급한 적 없는 임의의 가짜 `Mcp-Session-Id`로
# 생성한 클라이언트
fake_sid = str(uuid.uuid4())
mcp_fake = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=fake_sid)
r = mcp_fake.rpc_raw("tools/list")
print(f"FAKE Mcp-Session-Id={fake_sid}  -> HTTP {r.status_code}  body={r.text[:120]!r}")

## 9단계: 세션은 인증된 자격 증명 범위로 제한됨

세션은 인증된 사용자 범위로 제한됩니다. AgentCore Gateway는 인증 컨텍스트, OAuth 인그레스의 JWT bearer token 또는 AWS_IAM 인그레스의 IAM 자격 증명에서 사용자 자격 증명을 도출하고, 세션 내의 모든 요청이 동일한 사용자에게서 시작되었는지 검증합니다.

실제 동작: 다른 Cognito M2M 클라이언트(JWT의 `client_id` 클레임이 다름)는 동일한 게이트웨이에 유효한 토큰이 있더라도 다른 클라이언트의 `Mcp-Session-Id`를 재사용할 수 없습니다. 서로 다른 자격 증명 간에 세션을 재사용하면 HTTP 404 `Session not found or expired`가 반환됩니다. 게이트웨이는 세션이 다른 사용자에게 속한다는 정보를 노출하는 대신 해당 세션이 존재하지 않는 것처럼 처리합니다.

# 10단계: AgentCore Runtime 대상의 응답 속도 향상


대상의 세션을 재사용하면 AgentCore Runtime이 요청마다 새로운 MCP 서버 연결을 콜드 스타트할 필요가 없어 응답 시간이 단축됩니다.

첫 번째 호출과 이후 호출 간의 시간 차이를 확인합니다. 


In [ ]:
mcp_c = GatewayMCPClient(gatewayURL, _get_inbound_token)
mcp_c.initialize(client_info={"name": "session-demo-c", "version": "0.1"})
print(f"Session C id: {mcp_c.session_id}")
print()

In [ ]:
call_session_counter(
    mcp_c, "C.1"
)  # 첫 번째 호출 - AgentCore Runtime(MCP 대상) micro VM 생성

In [ ]:
call_session_counter(
    mcp_c, "C.2"
)  # 두 번째 호출 - AgentCore Runtime(MCP 대상) micro VM 재사용

## 11단계: 리소스 정리

아래 셀의 주석을 해제하여 이 Notebook에서 생성한 게이트웨이, OAuth2 자격 증명 공급자, MCP 서버 런타임 및 IAM 역할을 삭제합니다. Cognito CloudFormation 스택은 여러 실습에서 공유되므로 모든 실습을 마친 경우가 아니라면 그대로 둡니다.

In [ ]:
# utils.delete_gateway(gateway_client, gatewayID)

In [ ]:
# identity_client.delete_oauth2_credential_provider(name=f"{GATEWAY_NAME}-identity")

In [ ]:
# !cd mcpservers && agentcore remove agent --name {MCP_SERVER_NAME} -y
# !cd mcpservers && agentcore deploy -y

In [ ]:
# # ## 다른 실습에서 이 스택을 사용하지 않을 때 Cognito 스택 삭제
# print(f"Deleting stack {COGNITO_STACK_NAME}...")
# cfn.delete_stack(StackName=COGNITO_STACK_NAME)
# cfn.get_waiter("stack_delete_complete").wait(StackName=COGNITO_STACK_NAME)
# print(f"✅ Stack {COGNITO_STACK_NAME} deleted")

In [ ]:
utils.delete_iam_role(f"agentcore-{GATEWAY_NAME}-role")